# VGAE + SPC para detección de ataques en BATADAL — pipeline completo en Colab

| Paso | Qué hace | Salida |
|---|---|---|
| A | Modelo de referencia (hiperparámetros por defecto) + cartas SPC (Shewhart, EWMA, CUSUM, T², SPE) | `results/` |
| B | Búsqueda con Optuna: TPE + búsqueda aleatoria de control, varios workers en la GPU | `results/optuna_colab/` |
| C | Validación con semillas (primer uso del test) + reporte de evidencia | `results/optuna_colab/REPORTE.md` |
| D | Modelo final con los mejores hiperparámetros: curva de pérdida y cartas SPC | `results/optuna_colab_final/` |
| E | Figuras: grafo, arquitectura, progreso de Optuna, nodos anómalos por ataque | `results/figuras_optuna_colab/` |

**Protocolo:** el VGAE se entrena sólo con operación normal (`dataset03`); `dataset04` se usa para elegir hiperparámetros; el **test** se evalúa recién en el paso C. Mismo presupuesto de entrenamiento en todos los pasos: en GPU hasta 300 épocas con early stopping (paciencia 25), y la búsqueda explora batches de 128–512 para darle más trabajo a la GPU en cada paso.

**Uso de la GPU:** el modelo es pequeño, así que la GPU rinde más cuanto más trabajo recibe en paralelo. Si la utilización que muestra la celda de seguimiento se queda baja, sube `N_WORKERS` (prueba `os.cpu_count() + 1` o `+ 2`) y relanza: cada worker ocupa poca memoria de GPU.

**Antes de empezar**
1. Sube `batadal_colab.zip` a `MyDrive/Batadal_colab/`.
2. *Entorno de ejecución → Cambiar tipo de entorno → GPU* (T4, L4 o A100).
3. Ejecuta las celdas en orden. Todo se respalda en Drive: si Colab se desconecta, vuelve a ejecutar desde el principio y cada paso retoma o se salta si ya terminó.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Configuración

In [ ]:
import os
DRIVE_DIR = "/content/drive/MyDrive/Batadal_colab"   # carpeta en Drive con batadal_colab.zip
# Un worker por vCPU: con un modelo tan pequeño el cuello de botella es la CPU que lanza
# las operaciones a la GPU, no la GPU. Colab estándar suele dar 2 vCPU.
N_WORKERS = os.cpu_count()
N_TPE     = 100    # trials totales del estudio TPE
N_RANDOM  = 50     # trials totales de la búsqueda aleatoria (control)
N_SEEDS   = 10     # semillas para la validación de robustez
OUT       = "results/optuna_colab"

from google.colab import drive
drive.mount("/content/drive")

## Preparar el proyecto y restaurar lo que ya esté en Drive

In [ ]:
import os, json, shutil, subprocess, glob, time
os.makedirs(DRIVE_DIR, exist_ok=True)
# salir de /content/Batadal antes de borrarla (si no, la terminal queda en un directorio inexistente)
os.chdir("/content")
_running = subprocess.run(["pgrep", "-f", "vgae_spc.tune"], capture_output=True, text=True).stdout.split()
if _running:
    # hay workers trabajando en /content/Batadal: no se borra su carpeta
    print(f"hay {len(_running)} procesos de vgae_spc.tune corriendo: se conserva /content/Batadal tal cual")
else:
    !rm -rf /content/Batadal && mkdir -p /content/Batadal
    !unzip -q -o "{DRIVE_DIR}/batadal_colab.zip" -d /content/Batadal
!pip install -q optuna
os.chdir("/content/Batadal")

# resultados de una sesión anterior (modelo de referencia, semillas, modelo final, figuras)
if os.path.isdir(f"{DRIVE_DIR}/results") and not _running:
    shutil.copytree(f"{DRIVE_DIR}/results", "results", dirs_exist_ok=True)
    print("resultados restaurados desde Drive:", sorted(os.listdir("results")))
os.makedirs(OUT, exist_ok=True)

JOURNAL = f"{OUT}/journal.log"
backup = f"{DRIVE_DIR}/journal.log"
if os.path.exists(backup) and not _running:
    # se descartan líneas incompletas (la copia pudo hacerse a mitad de una escritura)
    good = []
    for line in open(backup):
        try:
            json.loads(line); good.append(line)
        except json.JSONDecodeError:
            pass
    open(JOURNAL, "w").writelines(good)
    print(f"estudio Optuna restaurado: {len(good)} operaciones")

def save_results():
    """Copia results/ a Drive (sin el journal, que se respalda aparte)."""
    shutil.copytree("results", f"{DRIVE_DIR}/results", dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("journal.log"))

from IPython.display import Image, Markdown, display, clear_output

def alive(stage):
    # workers vivos de una etapa, buscados en el sistema: funciona aunque se haya reiniciado
    # el notebook o no exista la variable procs (los workers corren desacoplados)
    r = subprocess.run(["pgrep", "-f", f"vgae_spc.tune --stage {stage}"], capture_output=True, text=True)
    return len(r.stdout.split())

## A. Modelo de referencia (hiperparámetros por defecto) + cartas SPC

In [ ]:
if not os.path.exists("results/summary.csv"):
    !python -m vgae_spc.run --retrain 2>&1 | grep -v "^ep "
    save_results()
else:
    print("ya calculado (results/summary.csv)")
import pandas as pd
display(pd.read_csv("results/summary.csv"))
for f in ("training_loss", "spc_test"):
    if os.path.exists(f"results/{f}.png"):
        display(Image(f"results/{f}.png"))

## B. Búsqueda con Optuna

In [ ]:
# Trials que quedaron en RUNNING por una desconexión se marcan como FAIL
# (si no, contarían para el total y el estudio terminaría antes de tiempo)
import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
optuna.logging.set_verbosity(optuna.logging.WARNING)

def study_status():
    out = {}
    if not os.path.exists(JOURNAL):
        return out
    storage = JournalStorage(JournalFileBackend(JOURNAL))
    for name in ("vgae_tpe", "vgae_random"):
        try:
            st = optuna.load_study(study_name=name, storage=storage)
        except KeyError:
            continue
        out[name] = (storage, st)
    return out

for name, (storage, st) in study_status().items():
    stale = [t for t in st.trials if t.state == optuna.trial.TrialState.RUNNING]
    for t in stale:
        storage.set_trial_state_values(t._trial_id, optuna.trial.TrialState.FAIL)
    done = sum(t.state == optuna.trial.TrialState.COMPLETE for t in st.trials)
    print(f"{name}: {done} completos, {len(stale)} marcados como FAIL")

In [ ]:
def launch(args, tag):
    # N_WORKERS procesos desacoplados del notebook: interrumpir una celda NO los detiene
    ps = []
    for i in range(N_WORKERS):
        log = open(f"{OUT}/{tag}_{i}.txt", "a")
        ps.append(subprocess.Popen(["python", "-m", "vgae_spc.tune", *args(i)], stdout=log,
                                   stderr=subprocess.STDOUT, start_new_session=True))
    return ps

if alive("search"):
    print(f"ya hay {alive('search')} workers de búsqueda corriendo; no se lanzan más")
else:
    procs = launch(lambda i: ["--stage", "search", "--tpe", str(N_TPE), "--random", str(N_RANDOM),
                              "--out", OUT, "--storage", JOURNAL, "--parallel"], "worker")
    print("workers lanzados:", [p.pid for p in procs])
    time.sleep(60)
    !head -n 1 {OUT}/worker_0.txt | grep -o "device: [a-z]*"   # debe decir: device: cuda
    !nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

### Seguimiento y respaldo

Cada 2 minutos se copia el estudio a Drive y se muestra el avance. Los workers corren desacoplados del notebook: puedes interrumpir esta celda (■) y volver a ejecutarla cuando quieras sin detenerlos. Para detenerlos de verdad: `!pkill -f vgae_spc.tune`.

In [ ]:
def sync():
    if os.path.exists(JOURNAL):
        tmp = f"{DRIVE_DIR}/journal.log.tmp"
        shutil.copy(JOURNAL, tmp); os.replace(tmp, f"{DRIVE_DIR}/journal.log")
    for f in glob.glob(f"{OUT}/worker_*.txt"):
        shutil.copy(f, DRIVE_DIR)

t0 = time.time()
while True:                      # muestra el estado de inmediato y luego cada 2 minutos
    sync()
    clear_output(wait=True)
    for name, n in (("vgae_tpe", N_TPE), ("vgae_random", N_RANDOM)):
        s = study_status().get(name)
        if s is None:
            print(f"{name:12s} aún no empieza"); continue
        done = [t for t in s[1].trials if t.state == optuna.trial.TrialState.COMPLETE]
        best = max((t.value for t in done), default=float("nan"))
        print(f"{name:12s} {len(done):4d}/{n} completos   mejor S_d04 = {best:.3f}")
    vivos = alive("search")
    print(f"{(time.time() - t0) / 60:.0f} min — workers vivos: {vivos}")
    !nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader
    if not vivos:
        break
    time.sleep(120)
sync()
print("no quedan workers de búsqueda corriendo")
!tail -n 3 {OUT}/worker_0.txt

## C. Validación con semillas (primer uso del test) y reporte de evidencia

In [ ]:
# Las 3 configuraciones × N_SEEDS semillas se reparten entre los workers. Cada resultado se
# guarda apenas termina (seeds_part_*.csv): si algo se corta, al relanzar se saltan los ya hechos.
if not os.path.exists(f"{OUT}/REPORTE.md"):
    if not alive("seeds"):
        seed_procs = launch(lambda i: ["--stage", "seeds", "--seeds", str(N_SEEDS), "--out", OUT,
                                       "--storage", JOURNAL, "--seed-worker", str(i),
                                       "--seed-workers", str(N_WORKERS)], "seeds")
    total = 3 * N_SEEDS
    while alive("seeds"):
        time.sleep(60)
        n = sum(max(len(open(f).readlines()) - 1, 0) for f in glob.glob(f"{OUT}/seeds_part_*.csv"))
        clear_output(wait=True); print(f"semillas: {n}/{total} entrenamientos")
        save_results()
    !python -m vgae_spc.tune --stage report --seeds {N_SEEDS} --out {OUT} --storage {JOURNAL} 2>&1 | grep -v Warning
    save_results()
display(Markdown(open(f"{OUT}/REPORTE.md").read()))
for f in ("history", "tpe_vs_random", "importances", "slices", "seeds"):
    display(Image(f"{OUT}/{f}.png"))

## D. Modelo final con los mejores hiperparámetros

In [ ]:
FINAL = OUT + "_final"
if not os.path.exists(f"{FINAL}/summary.csv"):
    !python -m vgae_spc.run --optuna --optuna-dir {OUT} --retrain 2>&1 | grep -v "^ep "
    save_results()
display(pd.read_csv(f"{FINAL}/summary.csv"))
for f in ("training_loss", "spc_test", "spc_dataset04", "sensor_contrib_test"):
    display(Image(f"{FINAL}/{f}.png"))

## E. Figuras: grafo, arquitectura, progreso de Optuna y nodos anómalos

In [ ]:
!python -m vgae_spc.figures --optuna-dir {OUT} 2>&1 | grep -v Warning
FIG = "results/figuras_optuna_colab"
for f in ("grafo_sensores", "arquitectura_vgae", "optuna_progreso", "nodos_anomalos_test"):
    display(Image(f"{FIG}/{f}.png"))
save_results()

## Descargar todo

In [ ]:
shutil.make_archive("/content/resultados_batadal", "zip", "results")
shutil.copy("/content/resultados_batadal.zip", DRIVE_DIR)
print("resultados en Drive:", f"{DRIVE_DIR}/resultados_batadal.zip")